# M04 — Joins y KPIs

[← Anterior](../M03-transformacion-datos/03-lab-reglas-negocio.ipynb) · [Siguiente →](02-lab-joins.ipynb)

Un KPI mentiroso casi siempre es un **join mal elegido**. Demo mínima con un huérfano.

Ejecuta las celdas **aquí**, en este mismo fichero. No lo copies a otro sitio.

Kernel: **Python (NovaShop)**.


## Arranque

Ejecuta estas dos celdas. Localizan el repo y dejan una `SparkSession` lista.


In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


In [ ]:
spark = get_spark('novashop-clase-m04')
print(spark.version, spark.sparkContext.master)


## Inner, left y anti

El inner **tira** al huérfano. El left lo deja. `left_anti` lo lista.


In [ ]:
from pyspark.sql import Row
from pyspark.sql.functions import col, sum as fsum, countDistinct

clientes = spark.createDataFrame([
    Row(customer_id="C1", country="ES"),
    Row(customer_id="C2", country="FR"),
])
lineas = spark.createDataFrame([
    Row(order_id="O1", customer_id="C1", gmv_line=100.0, is_billable=True),
    Row(order_id="O2", customer_id="C1", gmv_line=50.0, is_billable=True),
    Row(order_id="O3", customer_id="CX9", gmv_line=999.0, is_billable=True),
])
print("inner", lineas.join(clientes, "customer_id", "inner").count())
print("left ", lineas.join(clientes, "customer_id", "left").count())
lineas.join(clientes, "customer_id", "left_anti").show()


Ticket medio = `sum(GMV) / countDistinct(order_id)`, **no** `avg` de la línea.


In [ ]:
sales = lineas.join(clientes, "customer_id", "inner").where(col("is_billable"))
sales.agg(fsum("gmv_line").alias("gmv"), countDistinct("order_id").alias("orders")).show()
sales.groupBy("country").agg(fsum("gmv_line").alias("gmv")).show()


**Siguiente:** [lab de joins](02-lab-joins.ipynb) sobre el dataset real.
